# Milestone 3 — Training EN→ES Transformer on a T4Trains two capacity presets and compares greedy vs beam decoding.**Before running:** Runtime → Change runtime type → **T4 GPU**.Checkpoints are written to Google Drive after every epoch, so a killed sessionresumes rather than restarting.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csvimport torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Mount DriveCheckpoints survive session death only if they live here.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')DRIVE_ROOT = '/content/drive/MyDrive/nmt-en-es'CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'!mkdir -p "{CKPT_DIR}"print('checkpoints ->', CKPT_DIR)

## 2. Get the projectPick ONE. Option A is better — pushing to GitHub is 15 rubric points anyway.

In [ ]:
# --- Option A: clone from GitHub (recommended) ---# !git clone https://github.com/USERNAME/nmt-en-es.git /content/nmt-en-es# --- Option B: upload a zip of the project folder ---# from google.colab import files; files.upload()# !unzip -q nmt-en-es.zip -d /content/%cd /content/nmt-en-es!ls

In [ ]:
!pip install -q sentencepiece sacrebleu rouge-score 'datasets>=3.0'# torch is preinstalled on Colab; do NOT reinstall it.

## 3. Milestones 1–2Rebuild the corpus on Colab (faster download than a home connection) and re-runthe mask verification on GPU. Roughly 5 minutes.

In [ ]:
!python run_milestone1.py --force

In [ ]:
!python run_milestone2.py --steps 30

## 4. Train both presets`base` is 37.6M parameters, `small` is 24.0M. Dropout is held constant so thecomparison isolates capacity.Expect roughly 1–3 minutes per epoch on a T4. Early stopping halts each run oncevalidation loss stops improving, so this usually finishes well short of 20 epochs.**If the session dies, just re-run this cell** — it resumes from `last.pt`.

In [ ]:
!python run_milestone3.py --preset both --epochs 20 --checkpoint-dir "{CKPT_DIR}"

## 5. Verify the decodersCorrectness checks that BLEU cannot give you.

In [ ]:
!python verify_decoding.py --preset base  --checkpoint-dir "{CKPT_DIR}"!python verify_decoding.py --preset small --checkpoint-dir "{CKPT_DIR}"

## 6. Learning curvesThe epoch where validation loss turns upward while training loss keeps fallingis your overfitting point — name it explicitly in the report.

In [ ]:
import json, matplotlib.pyplot as pltfrom pathlib import Pathfig, axes = plt.subplots(1, 2, figsize=(12, 4.5))for ax, preset in zip(axes, ['base', 'small']):    p = Path(CKPT_DIR) / preset / 'history.json'    if not p.exists():        ax.set_title(f'{preset}: no history'); continue    h = json.loads(p.read_text())    ep = [r['epoch'] for r in h]    ax.plot(ep, [r['train_loss'] for r in h], 'o-', label='train')    ax.plot(ep, [r['val_loss'] for r in h], 's-', label='validation')    best = min(h, key=lambda r: r['val_loss'])    ax.axvline(best['epoch'], ls='--', c='grey', lw=1)    ax.annotate(f"best epoch {best['epoch']}\nval {best['val_loss']:.3f}",                (best['epoch'], best['val_loss']), textcoords='offset points',                xytext=(10, 20), fontsize=9)    ax.set_title(f"{preset}  ({len(h)} epochs)")    ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend(); ax.grid(alpha=.3)plt.tight_layout()plt.savefig('artifacts/learning_curves.png', dpi=150, bbox_inches='tight')plt.show()

## 7. Results table for the report

In [ ]:
import jsonr = json.loads(open('artifacts/milestone3_results.json').read())print(f"{'model':<8}{'params':>12}{'val loss':>10}{'epochs':>8}"      f"{'greedy':>9}{'beam':>9}{'gain':>8}{'beam s/s':>10}")print('-' * 74)for k, v in r.items():    g = v['decoding']['greedy']['metrics']    b = v['decoding']['beam']['metrics']    print(f"{k:<8}{v['parameters']:>12,}{v['best_val_loss']:>10.4f}"          f"{v['epochs_run']:>8}{g['bleu']:>9}{b['bleu']:>9}"          f"{b['bleu'] - g['bleu']:>+8.2f}{b['sents_per_sec']:>10}")for k, v in r.items():    print(f"\n--- {k} sample output ---")    for gh, bh in zip(v['decoding']['greedy']['samples'][:3],                      v['decoding']['beam']['samples'][:3]):        print(f"  greedy: {gh}\n  beam  : {bh}\n")

## 8. Copy artifacts back to DriveColab's local disk is wiped when the session ends.

In [ ]:
!cp -r artifacts "{DRIVE_ROOT}/"!ls "{DRIVE_ROOT}/artifacts"